### 1. Build the feature vector

In [ ]:
import pandas as pd

df = pd.read_csv("flyrank-machine-learning/data/raw/content_refresh_anonymized.csv")

df["is_declining"] = (df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]).astype(int)

feature_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate"]
df_clean = df.dropna(subset=feature_cols + ["clicks_last_30d", "is_declining"])
df_clean[feature_cols].describe()

### 2. Feature notes
- ctr: click-through rate over 90d window, available at prediction time
- avg_position: average search ranking position, available at prediction time
- engagement_rate / scroll_rate: on-site behavior signals, available at prediction time
- All four exist before the 30-day window we are predicting into — none are label-derived

### 3. The leakage hunt

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df_clean[feature_cols]
y = df_clean["is_declining"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
model_clean = RandomForestClassifier(random_state=42, n_estimators=100).fit(Xtr, ytr)
print("Without leak:", accuracy_score(yte, model_clean.predict(Xte)))

X_leak = df_clean[feature_cols + ["clicks_last_30d"]]
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak, y, test_size=0.25, random_state=42)
model_leak = RandomForestClassifier(random_state=42, n_estimators=100).fit(Xtr2, ytr2)
print("With leak (clicks_last_30d):", accuracy_score(yte2, model_leak.predict(Xte2)))

### 4. What I excluded and why
- Excluded: trend_direction, trend_pct — verified these are label-derived: 100% of is_declining=1 rows have trend_direction="down" (checked via groupby), meaning this column is computed from the same window as the target itself.
- Excluded: clicks_last_30d — shown above to inflate accuracy from 0.594 to 0.621.
- Excluded: content_id, client_id as predictors — used only for identification/grouping.